# Logistic regression, end to end

A complete binary classifier on real data: load, split, scale, fit, and read a report that says more than one number.

Scaling is not cosmetic here. `LogisticRegression` applies L2 regularization by default, and a penalty on coefficient size is only fair once the features share a scale. Note the order: the scaler is fitted on the training rows only, then *applied* to the test rows with `transform`. Calling `fit_transform` on the test set would let it learn from data the model is about to be judged on.

In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Load a binary classification dataset
data = load_breast_cancer()
X, y = data.data, data.target

# 2. Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 3. Scale features (essential for regularized logistic regression)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train Logistic Regression model
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

# 5. Evaluate
y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9737

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.95      0.96        43
           1       0.97      0.99      0.98        71

    accuracy                           0.97       114
   macro avg       0.97      0.97      0.97       114
weighted avg       0.97      0.97      0.97       114

## What the output is telling you

- **Accuracy is 0.9737, and that is the least interesting number here.** The dataset is imbalanced (71 benign to 43 malignant in the test split), so a model that guessed "benign" every time would already score 0.62.
- **Read the recall on class 0 instead: 0.95.** Class 0 is malignant. The model missed about 2 malignant cases out of 43. On a cancer screen, that error is not interchangeable with a false alarm, and accuracy averages the two together as though it were.
- **Precision and recall answer different questions.** Precision asks how often a positive call is right; recall asks how many of the real positives you caught. `classification_report` prints both per class because the trade-off between them is the actual decision.

## When to reach for this

Logistic regression is the first thing to try on a binary outcome with tabular features. It trains in milliseconds, the coefficients are inspectable, and it gives calibrated probabilities rather than bare labels — which matters when the threshold is a business decision rather than a default of 0.5.

Reach for something else when the boundary is genuinely non-linear, when you have strong feature interactions you would otherwise have to write out by hand, or when the features are raw text, audio or pixels.

## Extend this notebook

- Print `model.predict_proba` and move the threshold off 0.5. Watch recall on the malignant class rise as precision falls.
- Put the scaler and the model in a `Pipeline`, then cross-validate. The single split above is one draw.
- Set `class_weight="balanced"` and see which way the errors move.
- Inspect `model.coef_` against `data.feature_names`, remembering that a coefficient is not a causal effect.